
Objetivo
- Nesta etapa vamos transformar os dados prontos para análise em dados prontos para treinamento.

O que vamos fazer?
1. carregar o dataset validado
2. verificar tipos e valores ausentes
3. limpar colunas e converter tipos
4. separar features e alvo
5. dividir treino e teste
6. aplicar transformações para o modelo
7. salvar os dados preparados

Importante
- A preparação de dados é a etapa em que o modelo deixa de ver texto e valores inconsistentes para ver dados em formato útil.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Bibliotecas importadas com sucesso!")


Bibliotecas importadas com sucesso!


In [2]:
# Definir caminhos do projeto
try:
    notebook_dir = Path.cwd()
    if notebook_dir.name == 'notebooks':
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
except:
    project_root = Path.cwd()

input_path = project_root / 'data' / 'processed' / '01_raw_validated.csv'
output_dir = project_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Projeto: {project_root}')
print(f'Arquivo de entrada: {input_path}')


Projeto: c:\Temp\telco-churn-ml
Arquivo de entrada: c:\Temp\telco-churn-ml\data\processed\01_raw_validated.csv


In [3]:
# Carregar o dataset validado

df = pd.read_csv(input_path)
print(f"Formato do dataset: {df.shape}")
print(df.head())


Formato do dataset: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Strea

In [4]:
# Verificações iniciais

print('Tipos das colunas:')
print(df.dtypes)

print('\nValores ausentes:')
print(df.isnull().sum())

print('\nResumo do alvo Churn:')
print(df['Churn'].value_counts())


Tipos das colunas:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Valores ausentes:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
Paperles

In [5]:
# Limpeza inicial

clean_df = df.copy()

# Converter TotalCharges para numérico
clean_df['TotalCharges'] = pd.to_numeric(clean_df['TotalCharges'], errors='coerce')

# Remover identificador do cliente
clean_df = clean_df.drop(columns=['customerID'], errors='ignore')

# Remover duplicatas, se existirem
clean_df = clean_df.drop_duplicates()

print('Dataset após limpeza:')
print(clean_df.shape)
print(clean_df.head())


Dataset após limpeza:
(7021, 20)
   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   

In [6]:
# Separar features e alvo

# Yes = 1, No = 0
X = clean_df.drop(columns=['Churn'])
y = clean_df['Churn'].map({'Yes': 1, 'No': 0})

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('\nDistribuição da variável alvo:')
print(y.value_counts())


Features shape: (7021, 19)
Target shape: (7021,)

Distribuição da variável alvo:
Churn
0    5164
1    1857
Name: count, dtype: int64


In [7]:
# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Treino:', X_train.shape, y_train.shape)
print('Teste:', X_test.shape, y_test.shape)
print('\nProporção do alvo no treino:')
print(y_train.value_counts(normalize=True))


Treino: (5616, 19) (5616,)
Teste: (1405, 19) (1405,)

Proporção do alvo no treino:
Churn
0    0.735577
1    0.264423
Name: proportion, dtype: float64


In [8]:
# Identificar colunas numéricas e categóricas

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

print('Colunas numéricas:', numeric_features)
print('\nColunas categóricas:', categorical_features)


Colunas numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Colunas categóricas: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [9]:
# Criar o preprocessador

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Preprocessor criado com sucesso!')


Preprocessor criado com sucesso!


In [10]:
# Aplicar transformações

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Treino processado:', X_train_processed.shape)
print('Teste processado:', X_test_processed.shape)

print('\nPrimeiras 5 linhas da matriz transformada:')
print(X_train_processed[:5])


Treino processado: (5616, 45)
Teste processado: (1405, 45)

Primeiras 5 linhas da matriz transformada:
[[-0.44031529 -1.24133083  0.19316451 -0.94634921  0.          1.
   1.          0.          1.          0.          0.          1.
   1.          0.          0.          0.          1.          0.
   1.          0.          0.          1.          0.          0.
   1.          0.          0.          1.          0.          0.
   1.          0.          0.          1.          0.          0.
   1.          0.          0.          1.          0.          0.
   1.          0.          0.        ]
 [-0.44031529 -0.71107812  0.64735514 -0.43534467  1.          0.
   1.          0.          1.          0.          0.          1.
   0.          0.          1.          0.          1.          0.
   0.          0.          1.          1.          0.          0.
   0.          0.          1.          1.          0.          0.
   1.          0.          0.          1.          0.          0.


In [11]:
# Salvar dados preparados

# Salvar treino e teste processados em arquivos separados
pd.concat([
    pd.Series(y_train.values, name='Churn'),
    pd.DataFrame(X_train_processed)
], axis=1).to_csv(output_dir / '03_X_train_preprocessed.csv', index=False)

pd.concat([
    pd.Series(y_test.values, name='Churn'),
    pd.DataFrame(X_test_processed)
], axis=1).to_csv(output_dir / '04_X_test_preprocessed.csv', index=False)

print('Arquivos gerados:')
print(output_dir / '03_X_train_preprocessed.csv')
print(output_dir / '04_X_test_preprocessed.csv')
print('\nPróximo passo: etapa de treinamento do modelo.')


Arquivos gerados:
c:\Temp\telco-churn-ml\data\processed\03_X_train_preprocessed.csv
c:\Temp\telco-churn-ml\data\processed\04_X_test_preprocessed.csv

Próximo passo: etapa de treinamento do modelo.
